# 05 — So sánh ResNet và mô hình đa phương thức

Chạy trên GPU (Colab, hoặc Modal qua `cloud/modal_nb05.py`) sau Notebook 04. Nguồn chính: `20261104-071259-h100`.
Notebook chạy inference A/B/C trên dev/eval của run 04, chọn threshold trên dev và xuất so sánh. Không huấn luyện, không gọi Qwen và không đọc test.

**Metric chính là mAP**, vì mAP không phụ thuộc threshold. Macro-F1 phụ thuộc cách chọn threshold, nên được báo theo hai lưới:
- lưới mở rộng xuống `1e-8` (chính);
- lưới cũ `0.05..0.95` (để đối chiếu với run `20261109-083552-102172`).

**Hạn chế:**
- Checkpoint ResNet đã được chọn trên toàn PAMI val (bao gồm eval); full chọn trên dev.
- Full dùng threshold mà Notebook 04 chọn trên dev với lưới cũ, vì run 04 không lưu xác suất dev. Riêng Macro-F1 vì vậy nghiêng về ResNet.
- ResNet chỉ có một seed.
- Kết quả full gốc và full ba seed là hai nhóm kết quả khác nhau, được ghi riêng.

## 1. Mount Drive và đồng bộ code
Code mới trong `src/models/` cần có trên main trước khi chạy Colab.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import sys

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/Contextual Emotion Recognition"
)
REPO_URL = (
    "https://github.com/minh-1105/"
    "Explainable-Contextual-Emotion-Recognition-via-"
    "Text-Guided-Chain-of-Thought-Reasoning.git"
)
CODE_ROOT = Path("/content/contextual-emotion-reasoning")

assert DRIVE_ROOT.is_dir(), f"Không tìm thấy project trên Drive: {DRIVE_ROOT}"
if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(CODE_ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(CODE_ROOT), "pull", "--ff-only"], check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_ROOT / "requirements.txt")],
    check=True,
)
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print("Drive root:", DRIVE_ROOT)
print("Code root :", CODE_ROOT)

## 2. Cấu hình và tạo run mới
Để đường dẫn model là `None` để chọn export mới nhất; có thể điền đường dẫn tuyệt đối để khóa một export cụ thể. Mỗi lần chạy tạo thư mục mới.

In [ ]:
import csv, hashlib, json, platform, gc
from datetime import datetime, timezone
from collections import Counter
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from tqdm.auto import tqdm
from src.models.visual_baseline import VisualBaseline, EXPERIMENTS
from src.datasets.emotic_dataset import EmoticDataset, EMOTION_LABELS
from src.evaluation.emotion_metrics import (multilabel_metrics, select_per_class_thresholds,
    select_constant_top_k, constant_top_k_predictions, paired_image_bootstrap,
    threshold_boundary_report)

SOURCE_RUN_ID = "20261104-071259-h100"
EXPORT_OVERRIDES = {name: None for name in EXPERIMENTS}
LEGACY_THRESHOLD_GRID = [round(x, 2) for x in np.arange(.05, 1., .05)]
# Xác suất ResNet bão hòa gần 0/1: ở run 20261109-083552-102172, A có 26/26 lớp ở đáy
# lưới 0.05, và 33-43% mẫu dương trên dev của A/C có xác suất < 1e-4. Thêm các mức log
# xuống 1e-8 để điểm tối ưu nằm trong lưới; lớp vẫn chạm đáy được báo ở cell 5.
LOW_THRESHOLDS = [m * 10.0 ** e for e in range(-8, -2) for m in (1, 3)] + [.01, .02, .03, .04]
CONFIG = {"source_run_id": SOURCE_RUN_ID, "batch_size": 32, "num_workers": 2,
          "seed": 42, "bootstrap_resamples": 2000, "error_examples_per_direction": 10,
          "primary_metric": "map", "bootstrap_metrics": ["map", "macro_f1"],
          "threshold_grid": LOW_THRESHOLDS + LEGACY_THRESHOLD_GRID,
          "legacy_threshold_grid": LEGACY_THRESHOLD_GRID,
          "full_threshold_source": "Notebook 04, chọn trên dev với legacy_threshold_grid"}
SOURCE_DIR = DRIVE_ROOT / "outputs/qwen3_multimodal" / SOURCE_RUN_ID
RUN_DIR = DRIVE_ROOT / "outputs/multimodal_comparison" / datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "figures").mkdir()
device = torch.device("cuda")
assert torch.cuda.is_available(), "Hãy bật GPU Colab."
commit = subprocess.check_output(["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"], text=True).strip()

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def safe(value):
    if isinstance(value, dict): return {k: safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [safe(v) for v in value]
    if isinstance(value, np.ndarray): return safe(value.tolist())
    if isinstance(value, np.generic): return safe(value.item())
    if isinstance(value, float) and not np.isfinite(value): return None
    return value

def write_json(name, value):
    (RUN_DIR / name).write_text(json.dumps(safe(value), ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

def digest(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""): h.update(block)
    return h.hexdigest()

def write_csv(name, rows):
    fields = list(dict.fromkeys(k for row in rows for k in row))
    with (RUN_DIR / name).open("w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(safe(rows))

(RUN_DIR / "environment.txt").write_text(
    f"python={platform.python_version()}\ntorch={torch.__version__}\ncuda={torch.version.cuda}\ngpu={torch.cuda.get_device_name(0)}\ngit_commit={commit}\n", encoding="utf-8")
print("Run kết quả:", RUN_DIR)

## 3. Xác minh artifact H100 và ba export ResNet

In [ ]:
required_source = ["source_artifacts.json", "config.json", "environment.txt", "samples.json",
                   "metrics.json", "predictions_eval.npz", "comparison.json"]
assert all((SOURCE_DIR / name).is_file() for name in required_source), "Thiếu artifact nguồn H100."
manifest = read_json(SOURCE_DIR / "source_artifacts.json")
assert manifest["canonical_run_id"] == SOURCE_RUN_ID
source_config = read_json(SOURCE_DIR / "config.json")
assert source_config["category_policy"] == "union"
source_metrics = read_json(SOURCE_DIR / "metrics.json")
comparison = read_json(SOURCE_DIR / "comparison.json")
splits = read_json(SOURCE_DIR / "samples.json")["splits"]
assert all(len(splits[s]) == len(set(splits[s])) > 0 for s in ("dev", "eval"))
assert not set(splits["dev"]) & set(splits["eval"])
exports, configs = {}, {}
for name in EXPERIMENTS:
    candidates = [p for base in (DRIVE_ROOT / "models", DRIVE_ROOT / "checkpoints")
                  for p in (base / f"emotic_{name}_resnet50").glob("*") if (p / "model.pt").is_file()]
    assert EXPORT_OVERRIDES[name] or candidates, f"Thiếu export {name}."
    export = Path(EXPORT_OVERRIDES[name]) if EXPORT_OVERRIDES[name] else sorted(candidates, key=lambda p: (p.name, str(p)))[-1]
    cfg = read_json(export / "config.json")
    assert read_json(export / "label_map.json") == {label: i for i, label in enumerate(EMOTION_LABELS)}
    assert cfg["experiment_name"] == name and cfg["backbone"] == "resnet50"
    assert cfg["category_policy"] == "union" and cfg["image_size"] == 224
    assert np.isclose(cfg["crop_padding"], .05)
    exports[name], configs[name] = export, cfg
    print(name, export)
CONFIG["exports"] = {n: {"path": str(p), "sha256": digest(p / "model.pt")} for n,p in exports.items()}
write_json("config.json", CONFIG)
write_json("source_artifacts.json", {"source_dir": str(SOURCE_DIR), "manifest": manifest,
    "source_sha256": {n: digest(SOURCE_DIR / n) for n in required_source},
    "exports": CONFIG["exports"], "provisional": True,
    "note": "Manifest truy vết các stage; chưa xác minh artifact bằng hash tại thời điểm tạo model."})

## 4. Giải nén ảnh và khôi phục dev/eval
Chỉ mở annotation split val. Ghép mẫu bằng sample_id và kiểm tra ảnh không giao nhau.

In [ ]:
import shutil, time, zipfile
def extract_images(archive_path, extract_dir, root_name, expected_files):
    # Giải nén zip ảnh từ Drive vào ổ local của Colab, mỗi runtime một lần.
    image_root = extract_dir / root_name
    marker = extract_dir / ".extracted"
    if marker.is_file():
        return image_root
    assert archive_path.is_file(), f"Thiếu file zip: {archive_path}"
    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True)
    started = time.time()
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(extract_dir)
    file_count = sum(1 for path in image_root.rglob("*") if path.is_file())
    assert file_count == expected_files, (
        f"{archive_path.name}: {file_count} file, cần {expected_files}"
    )
    marker.touch()
    print(f"Giải nén {archive_path.name}: {file_count:,} file, {time.time() - started:.0f}s")
    return image_root


EMOTIC_ROOT = DRIVE_ROOT / "data/raw/raw/emotic"
ANNOTATION_PATH = EMOTIC_ROOT / "PAMI/Annotations/Annotations/Annotations.mat"
IMAGE_ARCHIVES = (
 (EMOTIC_ROOT / "PAMI/pami_emotic.zip", Path("/content/emotic/PAMI"), "emotic", 23185),
 (EMOTIC_ROOT / "CVPR/cvpr_emotic.zip", Path("/content/emotic/CVPR"), "cvpr_emotic", 18302))
IMAGE_ROOTS = tuple(extract_images(*spec) for spec in IMAGE_ARCHIVES)
transform = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize((.485,.456,.406), (.229,.224,.225))])
dataset = EmoticDataset(annotation_path=ANNOTATION_PATH, split="val", image_roots=IMAGE_ROOTS,
    category_policy="union", crop_padding=.05, strict_images=True,
    person_transform=transform, context_transform=transform)
lookup = {sample.sample_id: i for i,sample in enumerate(dataset.samples)}
assert set(splits["dev"]) | set(splits["eval"]) == set(lookup)
indices = {s: [lookup[sid] for sid in splits[s]] for s in ("dev", "eval")}
images = {s: [dataset.samples[i].relative_image_path for i in indices[s]] for s in indices}
assert not set(images["dev"]) & set(images["eval"])
targets = {s: np.stack([dataset._category_target(dataset.samples[i]).numpy() for i in indices[s]]) for s in indices}
loaders = {s: DataLoader(Subset(dataset, ix), batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=True) for s,ix in indices.items()}
for s in indices: print(s, len(indices[s]), "người /", len(set(images[s])), "ảnh")

## 5. Inference dev và khóa threshold trước eval
Chọn threshold theo lớp trên dev với cả lưới mở rộng (chính) và lưới cũ (đối chiếu). Lớp có threshold chạm biên lưới được đếm và cảnh báo: khi đó điểm tối ưu có thể nằm ngoài lưới và Macro-F1 bị đánh giá thấp.

In [ ]:
@torch.inference_mode()
def infer(name, split):
    cfg = configs[name]
    model = VisualBaseline(name, len(EMOTION_LABELS), cfg["freeze_backbone"], cfg["dropout"])
    model.load_state_dict(torch.load(exports[name] / "model.pt", map_location="cpu", weights_only=True), strict=True)
    model.to(device).eval()
    values = []
    for batch in tqdm(loaders[split], desc=f"{name} {split}"):
        values.append(model(batch["person_image"].to(device), batch["context_image"].to(device)).sigmoid().cpu().numpy())
    result = np.concatenate(values)
    assert result.shape == targets[split].shape and np.isfinite(result).all()
    del model
    gc.collect(); torch.cuda.empty_cache()
    return result

dev_prob = {n: infer(n, "dev") for n in EXPERIMENTS}
thresholds = {n: select_per_class_thresholds(targets["dev"], p, CONFIG["threshold_grid"]) for n,p in dev_prob.items()}
legacy_thresholds = {n: select_per_class_thresholds(targets["dev"], p, CONFIG["legacy_threshold_grid"])
                     for n,p in dev_prob.items()}
boundary = {n: threshold_boundary_report(targets["dev"], t, CONFIG["threshold_grid"]) for n,t in thresholds.items()}
legacy_boundary = {n: threshold_boundary_report(targets["dev"], t, CONFIG["legacy_threshold_grid"])
                   for n,t in legacy_thresholds.items()}
for n in thresholds:
    r, old = boundary[n], legacy_boundary[n]
    print(f"{n}: lưới mở rộng {r['at_min']}/{r['labels_with_positives']} lớp ở đáy, {r['at_max']} ở đỉnh;"
          f" lưới cũ {old['at_min']} lớp ở đáy")
    if r["at_min"] or r["at_max"]:
        print("  CẢNH BÁO: threshold chạm biên lưới mở rộng; điểm tối ưu có thể nằm ngoài lưới.")
constant = select_constant_top_k(targets["dev"])
write_json("selected_thresholds.json", {"selected_on": "dev",
    "grid": CONFIG["threshold_grid"], "thresholds": thresholds, "boundary": boundary,
    "legacy_grid": CONFIG["legacy_threshold_grid"], "legacy_thresholds": legacy_thresholds,
    "legacy_boundary": legacy_boundary, "constant_top_k": constant})
np.savez_compressed(RUN_DIR / "predictions_dev.npz", sample_ids=np.array(splits["dev"]),
    image_ids=np.array(images["dev"]), targets=targets["dev"], class_names=np.array(EMOTION_LABELS), **dev_prob)

## 6. Inference eval và ghép prediction full gốc
Full gốc trong metrics.json được giữ riêng với full ba seed trong comparison.json.

In [ ]:
eval_prob = {n: infer(n, "eval") for n in EXPERIMENTS}
with np.load(SOURCE_DIR / "predictions_eval.npz", allow_pickle=False) as data:
    ids = data["sample_ids"].astype(str).tolist()
    assert len(ids) == len(set(ids)) and set(ids) == set(splits["eval"])
    order_map = {sid: i for i,sid in enumerate(ids)}
    order = [order_map[sid] for sid in splits["eval"]]
    assert np.array_equal(data["targets"][order], targets["eval"]), "Nhãn full không khớp annotation."
    full_prob = data["probabilities"][order]
    full_thresholds = data["thresholds"].copy()
assert full_prob.shape == targets["eval"].shape and np.isfinite(full_prob).all()
assert full_thresholds.shape == (len(EMOTION_LABELS),)
metrics = {n: multilabel_metrics(targets["eval"], p, thresholds=thresholds[n], class_names=EMOTION_LABELS) for n,p in eval_prob.items()}
metrics["full_original"] = multilabel_metrics(targets["eval"], full_prob, thresholds=full_thresholds, class_names=EMOTION_LABELS)
legacy_metrics = {n: multilabel_metrics(targets["eval"], p, thresholds=legacy_thresholds[n], class_names=EMOTION_LABELS)
                  for n,p in eval_prob.items()}
# Notebook 04 chọn threshold full với lưới mặc định 0.05..0.95, trùng lưới cũ.
full_boundary = threshold_boundary_report(targets["dev"], full_thresholds, CONFIG["legacy_threshold_grid"])
print(f"full_original: {full_boundary['at_min']}/{full_boundary['labels_with_positives']} lớp ở đáy lưới 0.05 (threshold từ Notebook 04)")
for key in ("map", "macro_f1", "micro_f1"):
    assert np.isclose(metrics["full_original"][key], source_metrics["eval"][key]), f"Metric nguồn lệch: {key}"
constant_pred = constant_top_k_predictions(len(targets["eval"]), constant["label_order"], constant["k"], len(EMOTION_LABELS))
metrics["constant_top_k"] = multilabel_metrics(targets["eval"], predictions=constant_pred, class_names=EMOTION_LABELS)
scores = {**eval_prob, "full_original": full_prob}
model_thresholds = {**thresholds, "full_original": full_thresholds}
predictions = {n: scores[n] >= model_thresholds[n] for n in scores}
legacy_predictions = {n: p >= legacy_thresholds[n] for n,p in eval_prob.items()}
np.savez_compressed(RUN_DIR / "predictions_eval.npz", sample_ids=np.array(splits["eval"]),
    image_ids=np.array(images["eval"]), targets=targets["eval"], class_names=np.array(EMOTION_LABELS),
    full_original=full_prob, full_thresholds=full_thresholds, constant_predictions=constant_pred,
    **eval_prob, **{n+"_thresholds": t for n,t in thresholds.items()},
    **{n+"_legacy_thresholds": t for n,t in legacy_thresholds.items()})

## 7. Bootstrap và tổng hợp seed đã lưu
Full gốc được so với từng ResNet trên mAP (dùng xác suất, không phụ thuộc threshold) và Macro-F1 (lưới mở rộng), ghép cặp theo ảnh. Full-vs-C trên lưới cũ được tính lại để đối chiếu với run trước. Full-vs-ViT-only kế thừa từ một lần train khác trong comparison.json. Không gộp các CI thành cùng một model.

In [ ]:
bootstrap = {}
for name in EXPERIMENTS:
    for metric in CONFIG["bootstrap_metrics"]:
        a, b = ((scores["full_original"], scores[name]) if metric == "map"
                else (predictions["full_original"], predictions[name]))
        bootstrap[f"full_original_vs_{name}_{metric}"] = {"origin": "computed_notebook05",
            "threshold_grid": None if metric == "map" else "extended",
            "result": paired_image_bootstrap(targets["eval"], a, b, images["eval"],
                n_resamples=CONFIG["bootstrap_resamples"], seed=CONFIG["seed"], metric=metric)}
bootstrap["full_original_vs_person_context_macro_f1_legacy_grid"] = {"origin": "computed_notebook05",
    "threshold_grid": "legacy", "result": paired_image_bootstrap(targets["eval"],
        predictions["full_original"], legacy_predictions["person_context"], images["eval"],
        n_resamples=CONFIG["bootstrap_resamples"], seed=CONFIG["seed"], metric="macro_f1")}
bootstrap["full_seed42_vs_vit_only_seed42"] = {"origin": "inherited_comparison.json",
    "result": comparison["full_vs_vit_only"]}
write_json("bootstrap.json", bootstrap)
for key, item in bootstrap.items():
    r = item["result"]
    print(f"{key}: {r['difference_a_minus_b']:+.4f} [{r['ci95_lower']:+.4f}, {r['ci95_upper']:+.4f}]")

LIMITATIONS = ["Checkpoint ResNet chọn trên toàn PAMI val bao gồm eval; full chọn trên dev.",
    "ResNet có một seed; mean/SD ViT/full kế thừa Notebook 04.",
    "Full gốc và full ba seed là các run khác nhau; prediction từng mẫu của ViT-only không có.",
    "Nguồn stage từ manifest; hash hiện tại không chứng minh commit tạo từng artifact.",
    f"Full dùng threshold Notebook 04 chọn trên dev với lưới 0.05..0.95 vì run 04 không lưu xác suất dev; "
    f"{full_boundary['at_min']}/{full_boundary['labels_with_positives']} lớp ở đáy lưới. ResNet dùng lưới mở rộng "
    f"xuống {min(CONFIG['threshold_grid'])}, nên so sánh Macro-F1 nghiêng về ResNet; mAP là metric chính."]
write_json("metrics.json", {"scope": "PAMI val dev/eval, union, không test", "limitations": LIMITATIONS,
    "primary_metric": CONFIG["primary_metric"], "eval": metrics, "eval_legacy_grid": legacy_metrics,
    "threshold_boundary": {**boundary, "full_original": full_boundary}, "legacy_threshold_boundary": legacy_boundary,
    "inherited_comparison": comparison, "bootstrap": bootstrap})
rows = [{"model": n, "origin": "prediction_eval", "threshold_grid": "extended" if n in EXPERIMENTS else None,
         **{k:v for k,v in m.items() if k != "per_class"}} for n,m in metrics.items()]
rows += [{"model": n+"_legacy_grid", "origin": "prediction_eval", "threshold_grid": "legacy",
          **{k:v for k,v in m.items() if k != "per_class"}} for n,m in legacy_metrics.items()]
for name, vals in comparison["variants"].items():
    rows.append({"model": name+"_three_seeds", "origin": "comparison.json", **{k:v["mean"] for k,v in vals.items()},
                 **{k+"_sd": v["sd"] for k,v in vals.items()}})
write_csv("metrics.csv", rows)
write_csv("per_class_metrics.csv", [{"model": n, **r} for n,m in metrics.items() for r in m["per_class"]])

# AP theo lớp: lớp nào full hơn/thua ResNet, kèm support để thấy lợi ích có tập trung ở lớp hiếm không.
ap = {n: np.array([np.nan if r["ap"] is None else r["ap"] for r in metrics[n]["per_class"]], dtype=float)
      for n in scores}
ap_rows = []
for i, label in enumerate(EMOTION_LABELS):
    row = {"class_index": i, "class_name": label, "support_dev": int(targets["dev"][:, i].sum()),
           "support_eval": int(targets["eval"][:, i].sum()), "ap_full_original": ap["full_original"][i]}
    for name in EXPERIMENTS:
        row[f"ap_{name}"] = ap[name][i]
        row[f"delta_full_minus_{name}"] = ap["full_original"][i] - ap[name][i]
    ap_rows.append(row)
ap_rows.sort(key=lambda r: -np.nan_to_num(r["delta_full_minus_person_context"], nan=-np.inf))
write_csv("per_class_ap_delta.csv", ap_rows)

for row in rows:
    print(row["model"], "mAP:", row.get("map"), "macro-F1:", row.get("macro_f1"))

## 8. Phân tích lỗi, số người và biểu đồ

In [ ]:
import matplotlib.pyplot as plt
counts = Counter(s.relative_image_path for s in dataset.samples)
people = np.array([counts[x] for x in images["eval"]])
group_metrics = {}
for group, mask in {"1": people == 1, "2": people == 2, "3+": people >= 3}.items():
    if mask.any():
        group_metrics[group] = {n: multilabel_metrics(targets["eval"][mask], scores[n][mask],
                                    thresholds=model_thresholds[n], class_names=EMOTION_LABELS)
                                for n in scores}
write_json("people_group_metrics.json", group_metrics)
def sample_f1(p):
    den = targets["eval"].sum(1) + p.sum(1)
    return np.divide(2*(p*targets["eval"]).sum(1), den, out=np.zeros(len(den)), where=den > 0)
delta = sample_f1(predictions["full_original"]) - sample_f1(predictions["person_context"])
examples = {}
for name, sign in (("full_better", -1), ("resnet_better", 1)):
    chosen = sorted(range(len(delta)), key=lambda i: (sign*delta[i], splits["eval"][i]))[:CONFIG["error_examples_per_direction"]]
    examples[name] = [{"sample_id": splits["eval"][i], "relative_image_path": images["eval"][i],
        "delta_sample_f1": float(delta[i]), "target": targets["eval"][i].tolist(),
        "full_prediction": predictions["full_original"][i].tolist(),
        "resnet_prediction": predictions["person_context"][i].tolist()} for i in chosen]
write_json("error_examples.json", examples)
names = list(metrics)
x = np.arange(len(names)); fig, ax = plt.subplots(figsize=(11,5))
for j,key in enumerate(("map", "macro_f1", "micro_f1")):
    ax.bar(x+(j-1)*.25, [metrics[n][key] for n in names], .25, label=key)
ax.set_xticks(x, names, rotation=20, ha="right"); ax.set_ylim(0,1)
ax.set_title("PAMI val eval — nhãn union"); ax.legend(); fig.tight_layout()
fig.savefig(RUN_DIR / "figures/comparison.png", dpi=180); plt.show()
required = ["config.json", "environment.txt", "source_artifacts.json", "selected_thresholds.json", "metrics.json",
    "metrics.csv", "per_class_metrics.csv", "per_class_ap_delta.csv", "bootstrap.json", "predictions_dev.npz", "predictions_eval.npz",
    "error_examples.json", "people_group_metrics.json", "figures/comparison.png"]
assert all((RUN_DIR / p).is_file() for p in required)
print("Hoàn tất Notebook 05:", RUN_DIR)

## 9. Bàn giao

Trước khi kết luận, đọc `metrics.csv`, `bootstrap.json`, `per_class_ap_delta.csv`, `threshold_boundary` và các hạn chế trong `metrics.json`.

- **So ResNet với full: dựa trên mAP trước.** Macro-F1 chỉ là kết quả phụ, vì threshold của full vẫn theo lưới cũ của Notebook 04.
- **Còn lớp chạm biên lưới:** nếu `threshold_boundary` vẫn có lớp ở đáy, ghi rõ trong báo cáo rằng Macro-F1 của model đó có thể bị đánh giá thấp.

ViT-only chỉ có các metric tổng hợp đã lưu; cột thiếu không được suy ra từ mean/SD. Gửi thư mục run kết quả cho team. Notebook này chưa đánh giá test.